# 03 – BDT Training for Background Suppression

This notebook trains and evaluates a **Gradient Boosted Decision Tree (BDT)** classifier to suppress background in the semitauonic B decay analysis.

**Goals:**
- Prepare features and labels from MC samples
- Train an XGBoost BDT classifier
- Evaluate performance (ROC curve, AUC, feature importances)
- Choose an optimal working point
- Save the trained model for use in subsequent notebooks

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report
from xgboost import XGBClassifier

try:
    import mplhep as hep
    hep.style.use(hep.style.Belle2)
except ImportError:
    pass

plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

## 1. Load Data

In [ ]:
# ─── Paths – update as needed ────────────────────────────────────────────────
SIGNAL_FILE = "/path/to/signal_mc.root"
BKG_FILE    = "/path/to/background_mc.root"
TREE_NAME   = "ntuple"
# ─────────────────────────────────────────────────────────────────────────────

# Features to use in training – update based on EDA findings
FEATURES = [
    "M2_miss",
    "q2",
    "E_miss",
    "p_D_cms",
    "cos_theta_tau",
]

with uproot.open(SIGNAL_FILE) as f:
    sig = f[TREE_NAME].arrays(FEATURES, library="pd")
with uproot.open(BKG_FILE) as f:
    bkg = f[TREE_NAME].arrays(FEATURES, library="pd")

sig["label"] = 1
bkg["label"] = 0

data = pd.concat([sig, bkg], ignore_index=True).dropna(subset=FEATURES)
X = data[FEATURES]
y = data["label"]

print(f"Total samples : {len(data):,}  (signal: {y.sum():,}, bkg: {(~y.astype(bool)).sum():,})")

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

## 3. Train the BDT

In [ ]:
bdt = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)

bdt.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)

## 4. Evaluate Performance

In [ ]:
y_score = bdt.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
axes[0].plot(fpr, tpr, color="steelblue", label=f"AUC = {roc_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

# BDT output distributions
bins = np.linspace(0, 1, 50)
axes[1].hist(y_score[y_test == 1], bins=bins, density=True,
             alpha=0.6, label="Signal", color="steelblue")
axes[1].hist(y_score[y_test == 0], bins=bins, density=True,
             alpha=0.6, label="Background", color="tomato")
axes[1].set_xlabel("BDT score")
axes[1].set_ylabel("Normalized counts")
axes[1].set_title("BDT output distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("../docs/bdt_performance.pdf")
plt.show()
print(f"AUC = {roc_auc:.4f}")

## 5. Feature Importances

In [ ]:
importances = pd.Series(bdt.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(6, 4))
importances.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Importance (gain)")
ax.set_title("BDT feature importances")
plt.tight_layout()
plt.show()

## 6. Working Point Selection

Choose a cut on the BDT score that maximises signal significance S/√B.

In [ ]:
significance = np.where(
    (tpr * len(y_test[y_test == 1]) + (1 - fpr) * len(y_test[y_test == 0])) > 0,
    tpr * len(y_test[y_test == 1]) /
        np.sqrt(np.maximum((1 - fpr) * len(y_test[y_test == 0]) + tpr * len(y_test[y_test == 1]), 1e-9)),
    0
)

best_idx = np.argmax(significance)
best_thr = thresholds[best_idx]
print(f"Optimal BDT threshold : {best_thr:.3f}")
print(f"Signal efficiency     : {tpr[best_idx]:.3f}")
print(f"Background rejection  : {1 - fpr[best_idx]:.3f}")

## 7. Save the Model

In [ ]:
MODEL_PATH = "../docs/bdt_model.pkl"
joblib.dump(bdt, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")